In [ ]:
# Cell 1 : setup imports and notebook constants
# BTC Capture Review for HackRF recordings under RF_Sentinel/recordings.
# This notebook does a coarse channel-energy pass and a bounded offline Classic access/LAP scan.
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RECORDINGS_DIR = Path('/home/jake/workspace/SDR/RF_Sentinel/recordings')
BTC_CHANNEL_FREQS_HZ = {ch: 2_402_000_000 + ch * 1_000_000 for ch in range(79)}
TARGET_MACS = ['34:C9:F0:88:B5:97', '8C:B0:E9:4C:48:C3', 'C2:D1:CA:FF:33:BE']


plt.style.use('default')
pd.set_option('display.max_rows', 120)
pd.set_option('display.max_columns', 20)

RECORDINGS_DIR


In [ ]:
# Cell 2 : discover recordings and build the capture table
def discover_recordings(recordings_dir: Path):
    rows = []
    for iq_path in sorted(recordings_dir.glob('btc_hackrf_*_*.iq')):
        meta_path = iq_path.with_suffix('.json')
        if not meta_path.exists():
            continue
        meta = json.loads(meta_path.read_text())
        match = re.search(r'(\d{8}T\d{6}Z)', iq_path.name)
        batch_timestamp = match.group(1) if match else str(meta.get('captured_at_utc', ''))
        rows.append({
            'iq_path': iq_path,
            'meta_path': meta_path,
            'capture': iq_path.name,
            'batch_timestamp': batch_timestamp,
            'center_freq_hz': int(meta['center_freq_hz']),
            'sample_rate_sps': int(meta['sample_rate_sps']),
            'duration_seconds': float(meta['duration_seconds']),
            'samples': int(meta['samples']),
            'lna_gain_db': int(meta['lna_gain_db']),
            'vga_gain_db': int(meta['vga_gain_db']),
            'captured_at_utc': str(meta['captured_at_utc']),
            'size_mb': iq_path.stat().st_size / (1024 * 1024),
        })
    frame = pd.DataFrame(rows).sort_values(['batch_timestamp', 'center_freq_hz'], ascending=[False, True]).reset_index(drop=True)
    return frame

captures = discover_recordings(RECORDINGS_DIR)
captures


In [ ]:
# Cell 3 : define IQ loading and per-channel power helpers
def load_iq_i8(path: Path, max_complex_samples=None):
    raw = np.fromfile(path, dtype=np.int8)
    if raw.size % 2:
        raw = raw[:-1]
    if max_complex_samples is not None:
        raw = raw[: max_complex_samples * 2]
    i = raw[0::2].astype(np.float32) / 128.0
    q = raw[1::2].astype(np.float32) / 128.0
    return (i + 1j * q).astype(np.complex64)

def channel_power_table(iq: np.ndarray, center_freq_hz: int, sample_rate_sps: int, fft_size: int = 65536):
    usable = (iq.size // fft_size) * fft_size
    if usable == 0:
        raise ValueError('not enough samples for FFT analysis')
    iq = iq[:usable]
    windows = iq.reshape(-1, fft_size)
    window_fn = np.hanning(fft_size).astype(np.float32)
    spectra = np.fft.fftshift(np.fft.fft(windows * window_fn, axis=1), axes=1)
    power = np.mean(np.abs(spectra) ** 2, axis=0)
    power_db = 10.0 * np.log10(power + 1e-12)
    freqs = np.fft.fftshift(np.fft.fftfreq(fft_size, d=1.0 / sample_rate_sps)) + center_freq_hz

    rows = []
    half_bw = 0.5e6
    for ch, freq_hz in BTC_CHANNEL_FREQS_HZ.items():
        if abs(freq_hz - center_freq_hz) > (sample_rate_sps / 2.0):
            continue
        mask = (freqs >= (freq_hz - half_bw)) & (freqs < (freq_hz + half_bw))
        if not np.any(mask):
            continue
        rows.append({
            'channel': ch,
            'freq_hz': int(freq_hz),
            'power_db': float(np.mean(power_db[mask])),
            'peak_db': float(np.max(power_db[mask])),
            'covered': True,
        })
    return pd.DataFrame(rows).sort_values('channel').reset_index(drop=True), freqs, power_db

def summarize_capture(row, preview_samples=4_000_000):
    iq = load_iq_i8(Path(row["iq_path"]), max_complex_samples=preview_samples)
    channel_df, freqs, power_db = channel_power_table(iq, int(row["center_freq_hz"]), int(row["sample_rate_sps"]))
    channel_df["capture"] = Path(row["iq_path"]).name
    channel_df["center_freq_hz"] = int(row["center_freq_hz"])
    channel_df["sample_rate_sps"] = int(row["sample_rate_sps"])
    return channel_df, freqs, power_db


In [ ]:
# Cell 4 : run the coarse channel-energy analysis across all captures
analysis_rows = []
spectra_by_capture = {}
for _, row in captures.iterrows():
    channel_df, freqs, power_db = summarize_capture(row)
    analysis_rows.append(channel_df)
    spectra_by_capture[Path(row["iq_path"]).name] = {
        'freqs': freqs,
        'power_db': power_db,
    }

channel_power = pd.concat(analysis_rows, ignore_index=True)
channel_power.head()


In [ ]:
# Cell 5 : show the hottest BTC channels in each recording
top_channels = (
    channel_power.sort_values(['capture', 'power_db'], ascending=[True, False])
    .groupby('capture', as_index=False)
    .head(12)
    .reset_index(drop=True)
)

top_channels[['capture', 'channel', 'freq_hz', 'power_db', 'peak_db']]


In [ ]:
# Cell 6 : plot per-capture BTC channel power bars
capture_names = list(channel_power['capture'].drop_duplicates())
fig, axes = plt.subplots(len(capture_names), 1, figsize=(16, 3.6 * len(capture_names)), sharex=True)
if len(capture_names) == 1:
    axes = [axes]

for ax, capture_name in zip(axes, capture_names):
    frame = channel_power[channel_power["capture"] == capture_name].sort_values("channel")
    colors = ['#1f77b4' if ch not in {0, 24, 78} else '#2ca02c' for ch in frame['channel']]
    ax.bar(frame['channel'], frame['power_db'], color=colors, width=0.85)
    ax.set_title(capture_name)
    ax.set_ylabel('dB')
    ax.grid(True, alpha=0.25)
    ax.set_xlim(-1, 79)

axes[-1].set_xlabel('BTC channel')
plt.tight_layout()
plt.show()


In [ ]:
# Cell 7 : plot the BTC channel power heatmap
heatmap = (
    channel_power.pivot_table(index='capture', columns='channel', values='power_db', aggfunc='mean')
    .sort_index()
)

plt.figure(figsize=(18, 4.8))
plt.imshow(heatmap.values, aspect='auto', cmap='viridis')
plt.colorbar(label='Mean channel power (dB)')
plt.yticks(range(len(heatmap.index)), heatmap.index)
plt.xticks(range(len(heatmap.columns)), heatmap.columns)
plt.xlabel('BTC channel')
plt.ylabel('Capture')
plt.title('BTC channel power heatmap across HackRF captures')
plt.tight_layout()
plt.show()

heatmap


In [ ]:
# Cell 8 : define offline Classic access and LAP helpers
# This bounded offline scan reuses the same Classic preamble, Barker, and access-word ideas from the live decoder.
BTC_P = 0x83848D96BBCC54FC
BTC_GP = int("157464165547", 8)
BTC_G = (BTC_GP << 1) ^ BTC_GP
BTC_ACCESS_REPAIR_MAX_DISTANCE = 16

def bit_length(value: int) -> int:
    return int(value).bit_length()

def compute_remainder(input_value: int, divisor: int) -> int:
    divisor_length = bit_length(divisor)
    input_value <<= divisor_length
    while bit_length(input_value) >= divisor_length:
        input_value ^= divisor << (bit_length(input_value) - divisor_length)
    return input_value

def extract_lsb_byte(bits, start):
    value = 0
    for idx in range(8):
        value |= (bits[start + idx] & 1) << idx
    return value

def classic_barker(bits, pos):
    if pos + 70 >= len(bits):
        return None
    barker = extract_lsb_byte(bits, pos + 62) & 0x3F
    return barker if barker in {0x13, 0x2C} else None

def classic_expected_access_word(lap: int) -> int:
    barker_true = 0x13 if (lap & 0x800000) else 0x2C
    x = (barker_true << 24) | lap
    xtilde = (BTC_P >> 34) ^ x
    ctilde = compute_remainder(xtilde, BTC_G)
    return (ctilde | (xtilde << 34)) ^ BTC_P

def classic_access_candidate(bits, pos):
    barker = classic_barker(bits, pos)
    if barker is None:
        return None
    lap = ((extract_lsb_byte(bits, pos + 54) << 16) | (extract_lsb_byte(bits, pos + 46) << 8) | extract_lsb_byte(bits, pos + 38))
    code = (((extract_lsb_byte(bits, pos + 4) << 0) | (extract_lsb_byte(bits, pos + 12) << 8) | (extract_lsb_byte(bits, pos + 20) << 16) | (extract_lsb_byte(bits, pos + 28) << 24) | (extract_lsb_byte(bits, pos + 36) << 32)) & 0x3FFFFFFFF)
    observed = (barker << 58) | (lap << 34) | code
    expected = classic_expected_access_word(lap)
    distance = int((observed ^ expected).bit_count())
    return {
        "lap": lap,
        "observed_access_word": observed,
        "expected_access_word": expected,
        "distance": distance,
        "repaired": distance > 0 and distance <= BTC_ACCESS_REPAIR_MAX_DISTANCE,
    }

def classic_preamble_ok(bits, pos):
    if pos + 68 >= len(bits):
        return False
    even = bits[pos] + bits[pos + 2]
    odd = bits[pos + 1] + bits[pos + 3]
    return (even == 2 and odd == 0) or (even == 0 and odd == 2)

def gfsk_bits_1msps(z):
    prev = np.empty_like(z)
    prev[0] = np.complex64(1.0 + 0j)
    prev[1:] = z[:-1]
    cross = (prev.real * z.imag) - (prev.imag * z.real)
    cross = cross.astype(np.float32)
    cross -= float(np.median(cross))
    return [1 if v > 0 else 0 for v in cross.tolist()]

def design_lowpass_taps(sample_rate_hz: int, cutoff_hz: float, num_taps: int) -> np.ndarray:
    taps = max(15, int(num_taps) | 1)
    nyquist = max(1.0, float(sample_rate_hz) / 2.0)
    normalized_cutoff = min(0.98, max(0.001, float(cutoff_hz) / nyquist))
    n = np.arange(taps, dtype=np.float64) - ((taps - 1) / 2.0)
    kernel = normalized_cutoff * np.sinc(normalized_cutoff * n)
    kernel *= np.hamming(taps)
    kernel /= float(np.sum(kernel))
    return kernel.astype(np.float32)

def decimate_classic_lane(iq, center_freq_hz, sample_rate_sps, channel_freq_hz, lane_rate_sps=1_000_000):
    n = np.arange(iq.size, dtype=np.float32)
    offset_hz = float(channel_freq_hz - center_freq_hz)
    rot = np.exp((-2j * np.pi * offset_hz / float(sample_rate_sps)) * n).astype(np.complex64)
    mixed = iq * rot
    decim = max(1, int(round(sample_rate_sps / lane_rate_sps)))
    taps = design_lowpass_taps(sample_rate_sps, min(lane_rate_sps * 0.40, 800_000.0), max(31, min(193, (decim * 4) | 1)))
    filtered_i = np.convolve(mixed.real.astype(np.float32, copy=False), taps, mode="valid")
    filtered_q = np.convolve(mixed.imag.astype(np.float32, copy=False), taps, mode="valid")
    filtered = (filtered_i + 1j * filtered_q).astype(np.complex64)
    if decim <= 1:
        return filtered
    usable = (filtered.size // decim) * decim
    return filtered[:usable:decim].astype(np.complex64, copy=False)
def mac_to_target(mac: str):
    clean = re.sub(r'[^0-9A-Fa-f]', '', mac).upper()
    if len(clean) != 12:
        raise ValueError(f'invalid MAC {mac}')
    return {
        'mac': ':'.join(clean[i:i+2] for i in range(0, 12, 2)),
        'nap': clean[:4],
        'uap': clean[4:6],
        'lap': clean[6:],
        'lap_int': int(clean[6:], 16),
    }

def target_table(macs):
    return pd.DataFrame([mac_to_target(mac) for mac in macs])
def classic_header_from_repetition3(header_bits):
    header = 0
    perfect_rx = 0
    for idx in range(0, 54, 3):
        triple = header_bits[idx:idx + 3]
        s1 = sum(triple)
        s0 = 3 - s1
        header >>= 1
        if s1 == 0 or s0 == 0:
            perfect_rx += 1
        if s1 > s0:
            header |= 0x20000
    return header, perfect_rx

def classic_header_clks_for_uap(header: int, uap: int):
    found = []
    for clk in range(64):
        header_dewhiten = header
        whitener = (clk & 0x3F) | 0x40
        for bit_idx in range(18):
            whitener_out = (whitener >> 6) & 0x1
            whitener_shifted = (whitener << 1) & 0x7F
            whitener = whitener_shifted ^ (whitener_out | (whitener_out << 4))
            header_dewhiten ^= whitener_out << bit_idx

        lfsr = uap
        for bit_idx in range(10):
            lfsr_out = (lfsr >> 7) & 0x1
            data_in = (header_dewhiten >> bit_idx) & 0x1
            lfsr_in = lfsr_out ^ data_in
            lfsr_adder = ((lfsr_in << 7) | (lfsr_in << 5) | (lfsr_in << 2) | (lfsr_in << 1) | (lfsr_in << 0))
            lfsr = ((lfsr << 1) & 0xFF) ^ lfsr_adder

        for bit_idx in range(8):
            bit_rx = (header_dewhiten >> (10 + bit_idx)) & 0x1
            bit_tx = (lfsr >> (7 - bit_idx)) & 0x1
            if bit_rx != bit_tx:
                break
        else:
            found.append(clk)
    return found

def classic_bruteforce_uaps_reference(header_bits):
    header, perfect_rx = classic_header_from_repetition3(header_bits)
    if perfect_rx != 18:
        return {
            'header': header,
            'perfect_triplets': perfect_rx,
            'valid_uaps': 0,
            'uap_results': [],
        }
    results = []
    for uap in range(256):
        clks = classic_header_clks_for_uap(header, uap)
        if clks:
            results.append({'uap': uap, 'uap_hex': f'{uap:02X}', 'clks': clks})
    return {
        'header': header,
        'perfect_triplets': perfect_rx,
        'valid_uaps': len(results),
        'uap_results': results,
    }

def observed_access_word_from_bits(bits, pos):
    barker = classic_barker(bits, pos)
    if barker is None:
        return None
    lap = ((extract_lsb_byte(bits, pos + 54) << 16) | (extract_lsb_byte(bits, pos + 46) << 8) | extract_lsb_byte(bits, pos + 38))
    code = (((extract_lsb_byte(bits, pos + 4) << 0) | (extract_lsb_byte(bits, pos + 12) << 8) | (extract_lsb_byte(bits, pos + 20) << 16) | (extract_lsb_byte(bits, pos + 28) << 24) | (extract_lsb_byte(bits, pos + 36) << 32)) & 0x3FFFFFFFF)
    return (barker << 58) | (lap << 34) | code


In [ ]:
# Cell 9 : auto-pick the newest capture batch and show the BTC targets we are looking for
top_n_channels = 12
offline_preview_complex_samples = 6_000_000

targets_df = target_table(TARGET_MACS)
latest_batch_timestamp = captures.iloc[0]["batch_timestamp"]
batch_captures = captures[captures["batch_timestamp"] == latest_batch_timestamp].sort_values("center_freq_hz").reset_index(drop=True)
capture_idx_within_batch = 0
capture_row = batch_captures.iloc[capture_idx_within_batch]
capture_name = Path(capture_row["iq_path"]).name
hot_channels = (
    channel_power[channel_power["capture"] == capture_name]
    .sort_values("power_db", ascending=False)
    .head(top_n_channels)[["channel", "freq_hz", "power_db", "peak_db"]]
    .reset_index(drop=True)
)

latest_batch_timestamp, batch_captures[["capture", "center_freq_hz", "captured_at_utc"]], targets_df, capture_name, hot_channels


In [ ]:
# Cell 10 : run the bounded offline Classic access and LAP scan with target checks
iq = load_iq_i8(Path(capture_row["iq_path"]), max_complex_samples=offline_preview_complex_samples)
center_freq_hz = int(capture_row["center_freq_hz"])
sample_rate_sps = int(capture_row["sample_rate_sps"])
target_laps = {row.lap: row.mac for row in targets_df.itertuples(index=False)}

offline_hits = []
for _, row in hot_channels.iterrows():
    ch = int(row["channel"])
    freq_hz = int(row["freq_hz"])
    lane = decimate_classic_lane(iq, center_freq_hz, sample_rate_sps, freq_hz)
    bits = gfsk_bits_1msps(lane)
    preamble_hits = 0
    barker_hits = 0
    access_hits = 0
    repaired_hits = 0
    best_distance = 68
    lap_counts = {}
    target_best_distance = 68
    target_lap_hits = {}
    min_packet_bits = 72 + 54
    for pos in range(0, max(0, len(bits) - min_packet_bits)):
        if not classic_preamble_ok(bits, pos):
            continue
        preamble_hits += 1
        candidate = classic_access_candidate(bits, pos)
        if candidate is None:
            continue
        barker_hits += 1
        best_distance = min(best_distance, int(candidate["distance"]))
        lap_hex = f"{int(candidate['lap']):06X}"
        for target_row in targets_df.itertuples(index=False):
            target_distance = int((candidate["observed_access_word"] ^ classic_expected_access_word(target_row.lap_int)).bit_count())
            target_best_distance = min(target_best_distance, target_distance)
        if candidate["distance"] == 0 or candidate["distance"] <= BTC_ACCESS_REPAIR_MAX_DISTANCE:
            access_hits += 1
            if candidate["repaired"]:
                repaired_hits += 1
            lap_counts[lap_hex] = lap_counts.get(lap_hex, 0) + 1
            if lap_hex in target_laps:
                target_lap_hits[lap_hex] = target_lap_hits.get(lap_hex, 0) + 1
    top_laps = sorted(lap_counts.items(), key=lambda item: item[1], reverse=True)[:8]
    offline_hits.append({
        "channel": ch,
        "freq_hz": freq_hz,
        "power_db": float(row["power_db"]),
        "preamble_hits": preamble_hits,
        "barker_hits": barker_hits,
        "access_hits": access_hits,
        "repaired_hits": repaired_hits,
        "best_distance": best_distance if best_distance < 68 else None,
        "target_best_distance": target_best_distance if target_best_distance < 68 else None,
        "target_lap_hits": target_lap_hits,
        "top_laps": top_laps,
    })

offline_df = pd.DataFrame(offline_hits).sort_values(["target_best_distance", "access_hits", "barker_hits", "power_db"], ascending=[True, False, False, False], na_position="last").reset_index(drop=True)
offline_df


In [ ]:
# Cell 11 : show the offline BTC channel summary with target-distance columns
display_cols = ["channel", "freq_hz", "power_db", "preamble_hits", "barker_hits", "access_hits", "repaired_hits", "best_distance", "target_best_distance", "target_lap_hits"]
offline_df[display_cols]


In [ ]:
# Cell 12 : show the top offline LAPs and flag target LAPs explicitly
lap_rows = []
for _, row in offline_df.iterrows():
    for lap, count in row["top_laps"]:
        lap_rows.append({
            "channel": int(row["channel"]),
            "freq_hz": int(row["freq_hz"]),
            "lap": lap,
            "count": int(count),
            "target_mac": target_laps.get(lap, ""),
            "is_target": lap in target_laps,
            "best_distance": row["best_distance"],
            "target_best_distance": row["target_best_distance"],
        })
lap_df = pd.DataFrame(lap_rows).sort_values(["is_target", "count", "target_best_distance", "best_distance"], ascending=[False, False, True, True]).reset_index(drop=True)
lap_df.head(60)


In [ ]:
# Cell 13 : run reference-style target LAP and UAP scan across the newest batch
# This cell uses the original btsniffer header rule: only 18/18 perfect header triplets are accepted.
target_scan_preview_complex_samples = 8_000_000
target_scan_max_distance = 16
target_scan_rows = []

for _, capture_row_scan in batch_captures.iterrows():
    iq_scan = load_iq_i8(Path(capture_row_scan["iq_path"]), max_complex_samples=target_scan_preview_complex_samples)
    center_freq_hz_scan = int(capture_row_scan["center_freq_hz"])
    sample_rate_sps_scan = int(capture_row_scan["sample_rate_sps"])
    covered_channels = [
        (ch, freq_hz)
        for ch, freq_hz in BTC_CHANNEL_FREQS_HZ.items()
        if abs(freq_hz - center_freq_hz_scan) <= (sample_rate_sps_scan / 2.0)
    ]
    for ch, freq_hz in covered_channels:
        lane = decimate_classic_lane(iq_scan, center_freq_hz_scan, sample_rate_sps_scan, freq_hz)
        bits_by_polarity = {0: gfsk_bits_1msps(lane)}
        bits_by_polarity[1] = [bit ^ 1 for bit in bits_by_polarity[0]]
        for polarity, bits in bits_by_polarity.items():
            min_packet_bits = 72 + 54
            for pos in range(0, max(0, len(bits) - min_packet_bits)):
                if not classic_preamble_ok(bits, pos):
                    continue
                observed_aw = observed_access_word_from_bits(bits, pos)
                if observed_aw is None:
                    continue
                for target in targets_df.itertuples(index=False):
                    expected_aw = classic_expected_access_word(int(target.lap_int))
                    distance = int((observed_aw ^ expected_aw).bit_count())
                    if distance > target_scan_max_distance:
                        continue
                    header_result = classic_bruteforce_uaps_reference(bits[pos + 72:pos + 72 + 54])
                    target_uap = int(target.uap, 16)
                    target_uap_clks = classic_header_clks_for_uap(header_result["header"], target_uap) if header_result["perfect_triplets"] == 18 else []
                    target_scan_rows.append({
                        "capture": Path(capture_row_scan["iq_path"]).name,
                        "batch_timestamp": capture_row_scan["batch_timestamp"],
                        "target_mac": target.mac,
                        "target_lap": target.lap,
                        "target_uap": target.uap,
                        "channel": ch,
                        "freq_hz": int(freq_hz),
                        "polarity": polarity,
                        "bit_pos": pos,
                        "access_distance": distance,
                        "strict_access": distance == 0,
                        "header_perfect_triplets": int(header_result["perfect_triplets"]),
                        "valid_uaps": int(header_result["valid_uaps"]),
                        "target_uap_clks": target_uap_clks,
                        "target_uap_valid": bool(target_uap_clks),
                        "uap_results": header_result["uap_results"][:8],
                    })

target_scan_df = pd.DataFrame(target_scan_rows)
if not target_scan_df.empty:
    target_scan_df = target_scan_df.sort_values(["target_uap_valid", "strict_access", "access_distance", "header_perfect_triplets", "valid_uaps"], ascending=[False, False, True, False, False]).reset_index(drop=True)
target_scan_df


In [ ]:
# Cell 14 : summarize target scan hits by target and channel
if target_scan_df.empty:
    target_summary_df = pd.DataFrame(columns=["target_mac", "target_lap", "channel", "hits", "strict_hits", "best_distance", "best_header_triplets", "target_uap_hits"])
else:
    target_summary_df = (
        target_scan_df.groupby(["target_mac", "target_lap", "target_uap", "channel", "freq_hz"], as_index=False)
        .agg(
            hits=("access_distance", "size"),
            strict_hits=("strict_access", "sum"),
            best_distance=("access_distance", "min"),
            best_header_triplets=("header_perfect_triplets", "max"),
            max_valid_uaps=("valid_uaps", "max"),
            target_uap_hits=("target_uap_valid", "sum"),
        )
        .sort_values(["target_uap_hits", "strict_hits", "best_distance", "hits"], ascending=[False, False, True, False])
        .reset_index(drop=True)
    )
target_summary_df.head(80)


In [ ]:
# Cell 15 : notebook notes and next offline step
# The target scan above is intentionally closer to the original btsniffer code than the live research-mode decoder.
# A strict_access row means the access word matched the target LAP exactly.
# target_uap_valid means the strict 18/18 repeated header produced the expected UAP HEC/clock result.
# If strict target access is still absent, the next useful experiment is to add the same small timing/CFO sweep used in the live app to this offline target-only scan.
